# Stage 05-B: Asymmetric Gated Fusion — PhoBERT (article-level) + COOLANT image mean-pool

**Proposal B** of two fusion architectures for 3-class Vietnamese fake news detection.

## Architecture

```
ViFactCheck article  (Statement + Evidence)
         │
         ▼
 PhoBERT-base-v2 (frozen, Stage 3.9 checkpoint)
 [CLS] token → [B, 768]    ← article-level text representation
         │
   TextProjector
   Linear[768→256] + LayerNorm + ReLU
   h_t [B, 256]
         │
         ├─────────────────────────────────────────┐
         │                                         │
         │   COOLANT image_aligned_clip [n, 64]    │
         │   (n pairs per article)                 │
         │     ▼                                   │
         │   MEAN POOL over n pairs → [B, 64]      │
         │     ▼                                   │
         │   ImageProjector                        │
         │   Linear[64→256] + LayerNorm + ReLU     │
         │   h_v [B, 256]                          │
         │                                         │
         └──────── concat ──────────────────────────┘
                      │
              [B, 512]
                      ▼
    ┌─ Residual Gating Block ──────────────────────┐
    │  gate_logit = Linear[512→256]                │
    │  g = sigmoid(gate_logit)          [B, 256]   │
    │  fused = g * h_t + (1-g) * h_v   [B, 256]   │
    │  residual = fused + LayerNorm(h_t)           │  ← text residual keeps NEI signal
    └──────────────────────────────────────────────┘
                      │
         Dropout(0.3) + Linear[256→3]
                      │
         (Real=0, Fake=1, NEI=2)
```

## Differences vs Proposal A

| | Proposal A (05a) | Proposal B (05b) |
|---|---|---|
| Image aggregation | MIL attention (learned per-pair weights) | Mean pool (uniform) |
| Residual | None | Text residual added after gate |
| Params | ~200K | ~200K |
| Strength | Learns which image is most discriminative | Simpler, more stable; text residual protects NEI |
| Weakness | Attention can overfit on small # pairs | Mean pool loses per-pair discriminability |

**Inputs:**
- `training/stage39_features/stage39_{train,dev,test}.h5` — PhoBERT CLS [N_articles, 768]
- `training/stage2_features/stage2_{train,dev,test}.h5` — COOLANT image_aligned_clip [N_pairs, 64] + article_ids

**Outputs:**
- `training/checkpoints_stage05b/{YYYYMMDD_HHMMSS}_asym-gate_phobert768-coolant64_3cls_bs{B}_lr{lr}/best_model.pth`
- `training/stage05b_results/`


In [ ]:
# ─── Environment Setup (do not edit) ────────────────────────────────────────
import os, sys
from pathlib import Path

def _detect_platform():
    try:
        import google.colab  # noqa: F401
        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True

PLATFORM, _uncertain = _detect_platform()

if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
else:
    PROJECT_ROOT = _nb_path.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    "colab":   PROJECT_ROOT / ".env.colab",
    "vastai":  PROJECT_ROOT / ".env.vastai",
    "windows": PROJECT_ROOT / ".env.windows",
    "mac":     PROJECT_ROOT / ".env.mac",
}

if PLATFORM is None:
    _env_file = PROJECT_ROOT / ".env"
elif _uncertain:
    _env_file = _env_map["vastai"]
else:
    _env_file = _env_map[PLATFORM]

from dotenv import load_dotenv
if not _env_file.exists():
    _fallback = PROJECT_ROOT / ".env"
    if _fallback.exists():
        _env_file = _fallback
    else:
        raise FileNotFoundError(f"No .env file found. Expected: {_env_file}")
load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()

print(f"✅ Platform : {PLATFORM or 'unknown'}")
print(f"✅ Env file : {_env_file}")
print(f"✅ DATA_ROOT: {DATA_ROOT}")
print(f"{'✅' if DATA_ROOT.exists() else '❌'} Path exists: {DATA_ROOT.exists()}")

## Step 1: Configuration

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass
DATA_ROOT = Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT

CONFIG = {
    "paths": {
        "project_root": PROJECT_ROOT,
        # ── Inputs from previous stages ────────────────────────────────────
        # Stage 3.9: per-article PhoBERT CLS features [N_articles, 768]
        "stage39_features_dir": DATA_ROOT / "training" / "stage39_features",
        "stage39_checkpoint_root": DATA_ROOT / "training" / "checkpoints_vifactcheck",
        "stage39_manifest": None,  # None = auto-detect newest
        # Stage 2: per-pair COOLANT image_aligned_clip [N_pairs, 64] + article_ids
        "stage2_features_dir": DATA_ROOT / "training" / "stage2_features",
        # ── Outputs ────────────────────────────────────────────────────────
        # Checkpoint dirs named: {timestamp}_{arch-tag}_bs{B}_lr{lr}
        # Example: 20260617_143000_asym-gate_phobert768-coolant64_3cls_bs32_lr3e-4
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_stage05b",
        "results_dir":     DATA_ROOT / "training" / "stage05b_results",
        "mlflow_dir":      DATA_ROOT / "mlruns",
    },
    "model": {
        # Architecture tag — appears in checkpoint dir name
        "arch_tag": "asym-gate_phobert768-coolant64_3cls",
        "text_dim": 768,        # PhoBERT CLS dim (Stage 3.9)
        "image_dim": 64,        # COOLANT image_aligned_clip dim (Stage 2)
        "fusion_hidden": 256,   # projection hidden dim
        "num_classes": 3,       # Real=0, Fake=1, NEI=2
        "dropout": 0.3,
        # Whether to add text residual connection after gate (keeps NEI signal)
        "text_residual": True,
    },
    "training": {
        "batch_size": 32,      # articles per batch
        "max_epochs": 40,
        "patience": 8,         # early stop on val macro-F1
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.1,
        "grad_clip": 1.0,
        "onecycle_pct_start": 0.05,
        "seed": 42,
    },
    "mlflow": {
        "experiment_name": "stage05b-asymmetric-gated-fusion",
    },
    "safety": {
        "smoke_test": False,
        "smoke_batches": 3,
        "smoke_epochs": 2,
        "auto_install_deps": False,
        "force_rebuild_cache": False,
    },
}

CONFIG["paths"]["checkpoint_root"].mkdir(parents=True, exist_ok=True)
CONFIG["paths"]["results_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["paths"]["mlflow_dir"].mkdir(parents=True, exist_ok=True)

print(f"Project root:      {PROJECT_ROOT}")
print(f"Data root:         {DATA_ROOT}")
print(f"Stage39 features:  {CONFIG['paths']['stage39_features_dir']}")
print(f"Stage2  features:  {CONFIG['paths']['stage2_features_dir']}")
print(f"Checkpoint root:   {CONFIG['paths']['checkpoint_root']}")
print(f"Arch tag:          {CONFIG['model']['arch_tag']}")

## Step 2: Dependency Preflight, Device, Seed

In [ ]:
import importlib
_required = [("torch","torch"),("h5py","h5py"),("numpy","numpy"),("pandas","pandas"),
             ("matplotlib","matplotlib"),("seaborn","seaborn"),("tqdm","tqdm"),
             ("sklearn","scikit-learn"),("mlflow","mlflow")]
_missing = [pkg for mod, pkg in _required if importlib.util.find_spec(mod) is None]
if _missing:
    if CONFIG["safety"]["auto_install_deps"]:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install"] + _missing)
    else:
        raise RuntimeError(f"Missing packages: {_missing}")
else:
    print("All dependencies present.")

In [ ]:
import sys, random, json, gc
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def select_device():
    from src.utils.device import get_device
    return get_device()

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    print(f"Seed: {seed}")

device = select_device()
seed_everything(CONFIG["training"]["seed"])
print(f"Device: {device}")

## Step 3: Resolve Upstream Manifests

Same prerequisite check as Proposal A.

In [ ]:
def resolve_stage39_manifest(config):
    explicit = config["paths"]["stage39_manifest"]
    if explicit is not None and Path(explicit).exists():
        with open(explicit) as f:
            return json.load(f), Path(explicit)
    search_root = config["paths"]["stage39_checkpoint_root"]
    if not search_root.exists():
        return None, None
    candidates = list(search_root.rglob("checkpoint_manifest.json"))
    if not candidates:
        return None, None
    newest = max(candidates, key=lambda p: p.stat().st_mtime)
    with open(newest) as f:
        return json.load(f), newest

def validate_feature_cache(feat_dir, prefix, splits=("train", "dev", "test")):
    missing = [feat_dir / f"{prefix}_{s}.h5" for s in splits
               if not (feat_dir / f"{prefix}_{s}.h5").exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing feature cache files:\n" +
            "\n".join(f"  {p}" for p in missing) +
            "\nRun the prerequisite notebook first."
        )
    for s in splits:
        p = feat_dir / f"{prefix}_{s}.h5"
        with h5py.File(p, "r") as f:
            print(f"  [{prefix}_{s}] attrs={dict(f.attrs)}")
            for k in f.keys():
                print(f"    {k}: {f[k].shape}")

stage39_manifest, stage39_manifest_path = resolve_stage39_manifest(CONFIG)
if stage39_manifest is None:
    raise RuntimeError(
        "Stage 3.9 manifest not found.\n"
        "Run notebooks/pipeline/03.9_vifactcheck_training.ipynb first."
    )
print(f"✅ Stage 3.9 manifest : {stage39_manifest_path}")
print(f"   backbone           : {stage39_manifest.get('backbone')}")
print(f"   best_epoch         : {stage39_manifest.get('best_epoch')}")
print(f"   best val macro-F1  : {stage39_manifest.get('best_metrics', {}).get('val_macro_f1', '?')}")

s4_info = stage39_manifest.get("stage4_integration", {})
stage39_feat_dir = Path(s4_info.get("stage39_features_dir",
                                     str(CONFIG["paths"]["stage39_features_dir"])))
print(f"\n── Stage 3.9 features ({stage39_feat_dir.name}) ──")
validate_feature_cache(stage39_feat_dir, "stage39")

stage2_feat_dir = CONFIG["paths"]["stage2_features_dir"]
print(f"\n── Stage 2 features ({stage2_feat_dir.name}) ──")
validate_feature_cache(stage2_feat_dir, "stage2")

## Step 4: Build Article-Level Cache with Mean-Pooled Image Features

Unlike Proposal A (which pads to `max_pairs` for attention), Proposal B mean-pools all n image
features into one [64]-dim vector at cache-build time — simpler, no padding needed.

Output: `stage05b_{split}.h5` with:
- `text_features`  [N_articles, 768] — PhoBERT CLS
- `image_features` [N_articles, 64]  — mean-pooled COOLANT images
- `pair_counts`    [N_articles]      — number of pairs averaged
- `labels`         [N_articles]      — 3-class


In [ ]:
def build_article_cache(split, stage39_h5_path, stage2_h5_path, output_path,
                        force_rebuild=False):
    """Build article-level cache with mean-pooled COOLANT image features."""
    if Path(output_path).exists() and not force_rebuild:
        print(f"[article cache] {split}: using existing {output_path}")
        return

    print(f"Building article cache for {split}...")

    with h5py.File(stage39_h5_path, "r") as f:
        s39_text   = f["text_features"][:].astype(np.float32)   # [N_art, 768]
        s39_aids   = f["article_ids"][:].astype(np.int64)         # [N_art]
        s39_labels = f["labels"][:].astype(np.int64)              # [N_art]

    with h5py.File(stage2_h5_path, "r") as f:
        s2_image = f["image_features"][:].astype(np.float32)    # [N_pairs, 64]
        s2_aids  = f["article_ids"][:].astype(np.int64)          # [N_pairs]

    # Group Stage2 pairs by article_id → compute mean
    aid_to_imgs = defaultdict(list)
    for i, aid in enumerate(s2_aids):
        aid_to_imgs[int(aid)].append(s2_image[i])

    img_dim = s2_image.shape[1]  # 64

    text_out, image_out, pair_counts_out, labels_out, aids_out = [], [], [], [], []
    skipped_no_image = 0

    for i, art_id in enumerate(s39_aids):
        art_id = int(art_id)
        imgs = aid_to_imgs.get(art_id, [])
        if len(imgs) == 0:
            skipped_no_image += 1
            continue

        mean_img = np.mean(np.stack(imgs, axis=0), axis=0)  # [64]

        text_out.append(s39_text[i])
        image_out.append(mean_img)
        pair_counts_out.append(len(imgs))
        labels_out.append(s39_labels[i])
        aids_out.append(art_id)

    if skipped_no_image > 0:
        print(f"  ⚠  Skipped {skipped_no_image} articles with no Stage2 image pairs")

    text_arr  = np.stack(text_out,  axis=0)   # [N, 768]
    image_arr = np.stack(image_out, axis=0)   # [N, 64]
    cnt_arr   = np.array(pair_counts_out, dtype=np.int32)
    lbl_arr   = np.array(labels_out, dtype=np.int64)
    aid_arr   = np.array(aids_out,  dtype=np.int64)

    hist = np.bincount(lbl_arr, minlength=3)
    print(f"  Articles: {len(lbl_arr)}  |  label dist {hist.tolist()}")
    print(f"  Pair counts — min:{cnt_arr.min()}  mean:{cnt_arr.mean():.1f}  max:{cnt_arr.max()}")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(output_path, "w") as f:
        f.create_dataset("text_features",  data=text_arr)
        f.create_dataset("image_features", data=image_arr)
        f.create_dataset("pair_counts",    data=cnt_arr)
        f.create_dataset("labels",         data=lbl_arr)
        f.create_dataset("article_ids",    data=aid_arr)
        f.attrs["n_articles"]    = len(lbl_arr)
        f.attrs["text_dim"]      = 768
        f.attrs["image_dim"]     = img_dim
        f.attrs["num_classes"]   = 3
        f.attrs["image_pooling"] = "mean"
        f.attrs["label_hist"]    = str(hist.tolist())
    print(f"  Saved → {output_path}")


cache_dir = CONFIG["paths"]["checkpoint_root"].parent / "stage05b_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

for split in ["train", "dev", "test"]:
    build_article_cache(
        split=split,
        stage39_h5_path=stage39_feat_dir / f"stage39_{split}.h5",
        stage2_h5_path=stage2_feat_dir / f"stage2_{split}.h5",
        output_path=cache_dir / f"stage05b_{split}.h5",
        force_rebuild=CONFIG["safety"]["force_rebuild_cache"],
    )
print("\n✅ Article-level cache ready.")

## Step 5: Dataset and DataLoaders

In [ ]:
class ArticleDataset(Dataset):
    """Loads article-level cache: text [768], image [64] (mean-pooled), label."""

    def __init__(self, h5_path):
        with h5py.File(h5_path, "r") as f:
            self.text_features  = f["text_features"][:].astype(np.float32)   # [N, 768]
            self.image_features = f["image_features"][:].astype(np.float32)  # [N, 64]
            self.pair_counts    = f["pair_counts"][:].astype(np.int32)        # [N]
            self.labels         = f["labels"][:].astype(np.int64)             # [N]
        print(f"ArticleDataset: {len(self.labels)} articles")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.text_features[idx]),    # [768]
            torch.from_numpy(self.image_features[idx]),   # [64]
            int(self.labels[idx]),
        )


def create_dataloaders(config, cache_dir):
    bs    = config["training"]["batch_size"]
    smoke = config["safety"]["smoke_test"]
    smoke_n = config["safety"]["smoke_batches"]

    datasets = {
        s: ArticleDataset(cache_dir / f"stage05b_{s}.h5")
        for s in ["train", "dev", "test"]
    }
    raw_loaders = {
        "train": DataLoader(datasets["train"], batch_size=bs, shuffle=True,  num_workers=0),
        "dev":   DataLoader(datasets["dev"],   batch_size=bs, shuffle=False, num_workers=0),
        "test":  DataLoader(datasets["test"],  batch_size=bs, shuffle=False, num_workers=0),
    }
    if smoke:
        from itertools import islice
        class _Smoke:
            def __init__(self, l, n): self._l = l; self._n = n
            def __iter__(self): return islice(self._l, self._n)
            def __len__(self): return min(self._n, len(self._l))
        return {k: _Smoke(v, smoke_n) for k, v in raw_loaders.items()}, datasets
    return raw_loaders, datasets


loaders, datasets = create_dataloaders(CONFIG, cache_dir)

_b = next(iter(DataLoader(datasets["train"], batch_size=4, shuffle=False)))
_t, _img, _lbl = _b
print(f"Batch — text:{_t.shape}  image:{_img.shape}  labels:{_lbl}")
print(f"Train label hist: {np.bincount(datasets['train'].labels, minlength=3)}")

## Step 6: Asymmetric Gated Fusion Head with Text Residual

Architecture:
- `TextProjector`: Linear[768→256] + LayerNorm + ReLU
- `ImageProjector`: Linear[64→256] + LayerNorm + ReLU
- `AsymmetricGate`: sigmoid(Linear[512→256])
- `TextResidual`: LayerNorm(h_t) added after gate — preserves NEI signal
- `Classifier`: Dropout + Linear[256→3]


In [ ]:
class AsymmetricGatedFusionHead(nn.Module):
    """
    Asymmetric gated fusion with optional text residual for fake news detection.

    Inputs:
        text_feat  [B, 768] — frozen PhoBERT CLS (Stage 3.9)
        image_feat [B, 64]  — frozen COOLANT image_aligned_clip, mean-pooled (Stage 2)
    Output:
        logits [B, 3]  (Real=0, Fake=1, NEI=2)
    """

    def __init__(self, text_dim=768, image_dim=64, fusion_hidden=256,
                 num_classes=3, dropout=0.3, text_residual=True):
        super().__init__()
        self.text_residual = text_residual

        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, fusion_hidden),
            nn.LayerNorm(fusion_hidden),
            nn.ReLU(),
        )
        self.image_proj = nn.Sequential(
            nn.Linear(image_dim, fusion_hidden),
            nn.LayerNorm(fusion_hidden),
            nn.ReLU(),
        )
        # Gate: sigmoid(W[h_t; h_v]) → [256] blend coefficients
        # Asymmetric because text input (768→256) is richer than image (64→256)
        # but the gate treats both projections equally after projection
        self.gate = nn.Linear(fusion_hidden * 2, fusion_hidden)

        if text_residual:
            # LayerNorm on h_t before adding — stabilises residual scale
            self.residual_norm = nn.LayerNorm(fusion_hidden)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, num_classes),
        )

    def forward(self, text_feat, image_feat):
        h_t = self.text_proj(text_feat)    # [B, 256]
        h_v = self.image_proj(image_feat)  # [B, 256]

        g = torch.sigmoid(self.gate(torch.cat([h_t, h_v], dim=-1)))  # [B, 256]
        fused = g * h_t + (1.0 - g) * h_v                            # [B, 256]

        if self.text_residual:
            fused = fused + self.residual_norm(h_t)                   # [B, 256]

        return self.classifier(fused), g  # logits [B,3], gate [B,256] for inspection


# ── Instantiate and verify ────────────────────────────────────────────────
head = AsymmetricGatedFusionHead(
    text_dim=CONFIG["model"]["text_dim"],
    image_dim=CONFIG["model"]["image_dim"],
    fusion_hidden=CONFIG["model"]["fusion_hidden"],
    num_classes=CONFIG["model"]["num_classes"],
    dropout=CONFIG["model"]["dropout"],
    text_residual=CONFIG["model"]["text_residual"],
).to(device)

total_params    = sum(p.numel() for p in head.parameters())
trainable_params = sum(p.numel() for p in head.parameters() if p.requires_grad)
print(f"AsymmetricGatedFusionHead — total: {total_params:,}  trainable: {trainable_params:,}")

head.eval()
with torch.no_grad():
    _logits, _gate_vals = head(_t[:2].to(device), _img[:2].to(device))
print(f"Sanity — logits:{_logits.shape}  gate_mean:{_gate_vals.mean().item():.3f}")
print(f"  (gate > 0.5 → text wins; gate < 0.5 → image wins)")
print(f"  Gate mean across 256 dims = {_gate_vals.mean().item():.3f} (initially ~0.5)")

## Step 7: Loss, Optimizer, Scheduler

In [ ]:
train_labels = datasets["train"].labels
nc = CONFIG["model"]["num_classes"]
counts = np.bincount(train_labels, minlength=nc).astype(np.float64)
class_weights = torch.tensor(
    len(train_labels) / (nc * counts), dtype=torch.float32
).to(device)
print(f"Class counts : {counts.astype(int).tolist()}")
print(f"Class weights: {class_weights.cpu().tolist()}")

loss_fn = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CONFIG["training"]["label_smoothing"],
)

optimizer = torch.optim.AdamW(
    head.parameters(),
    lr=CONFIG["training"]["lr"],
    weight_decay=CONFIG["training"]["weight_decay"],
)

total_steps = CONFIG["training"]["max_epochs"] * len(loaders["train"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=CONFIG["training"]["lr"],
    total_steps=total_steps,
    pct_start=CONFIG["training"]["onecycle_pct_start"],
    anneal_strategy="cos",
)
print(f"Optimizer: AdamW  lr={CONFIG['training']['lr']}")
print(f"Scheduler: OneCycleLR  total_steps={total_steps}")

## Step 8: Checkpoint Naming and Save Helpers

Checkpoint directory name: `{YYYYMMDD_HHMMSS}_{arch_tag}_bs{B}_lr{lr}[_noresidual]`

Example: `20260617_150022_asym-gate_phobert768-coolant64_3cls_bs32_lr3e-4`

In [ ]:
def _lr_to_str(lr):
    return f"{lr:.0e}".replace("+0", "").replace("-0", "-").replace("e0","e")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
arch_tag  = CONFIG["model"]["arch_tag"]
bs        = CONFIG["training"]["batch_size"]
lr_str    = _lr_to_str(CONFIG["training"]["lr"])
residual_tag = "" if CONFIG["model"]["text_residual"] else "_noresidual"
run_name  = f"{timestamp}_{arch_tag}_bs{bs}_lr{lr_str}{residual_tag}"

if CONFIG["safety"]["smoke_test"]:
    run_name = f"SMOKE_{run_name}"

run_dir = CONFIG["paths"]["checkpoint_root"] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
best_ckpt_path = run_dir / "best_model.pth"

print(f"Run dir: {run_dir}")
print(f"Reading the name tells you:")
print(f"  - Date/time: {timestamp}")
print(f"  - Architecture: asymmetric gate, PhoBERT 768-dim text + COOLANT 64-dim image, mean-pooled")
print(f"  - 3-class output, text residual: {CONFIG['model']['text_residual']}")
print(f"  - Batch size: {bs}, LR: {lr_str}")


def _cfg_jsonable(cfg):
    if isinstance(cfg, dict):  return {k: _cfg_jsonable(v) for k, v in cfg.items()}
    if isinstance(cfg, Path):  return str(cfg)
    if isinstance(cfg, (list, tuple)): return [_cfg_jsonable(v) for v in cfg]
    return cfg


def save_checkpoint(path, model, epoch, metrics, config, history, is_best):
    torch.save({
        "model_state_dict":   model.state_dict(),
        "config":             _cfg_jsonable(config),
        "epoch":              epoch,
        "metrics":            metrics,
        "training_history":   history,
        "arch_tag":           config["model"]["arch_tag"],
        "text_dim":           config["model"]["text_dim"],
        "image_dim":          config["model"]["image_dim"],
        "fusion_hidden":      config["model"]["fusion_hidden"],
        "num_classes":        config["model"]["num_classes"],
        "text_residual":      config["model"]["text_residual"],
        "is_best":            is_best,
        "run_name":           run_name,
    }, path)


def write_manifest(run_dir, best_ckpt_path, best_epoch, best_metrics, config):
    manifest = {
        "notebook":             "05b_asymmetric_gated_fusion.ipynb",
        "proposal":             "B — Asymmetric Gated Fusion with Text Residual",
        "run_name":             run_name,
        "best_checkpoint_path": str(best_ckpt_path),
        "best_epoch":           best_epoch,
        "selection_metric":     "val_macro_f1",
        "best_metrics":         best_metrics,
        "arch": {
            "tag":           config["model"]["arch_tag"],
            "text_encoder":  "PhoBERT-base-v2 (frozen, Stage 3.9)",
            "text_dim":      config["model"]["text_dim"],
            "image_encoder": "COOLANT image_aligned_clip (frozen, Stage 2, mean-pooled)",
            "image_dim":     config["model"]["image_dim"],
            "fusion":        "asymmetric gate + text residual",
            "text_residual": config["model"]["text_residual"],
            "num_classes":   config["model"]["num_classes"],
        },
        "training": _cfg_jsonable(config["training"]),
    }
    p = run_dir / "checkpoint_manifest.json"
    with open(p, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest written: {p}")

print("Checkpoint helpers defined.")

## Step 9: Training Loop

In [ ]:
def compute_metrics(y_true, y_pred, prefix, nc):
    acc      = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    per_f1   = f1_score(y_true, y_pred, average=None, zero_division=0)
    m = {
        f"{prefix}_accuracy":  round(acc, 4),
        f"{prefix}_macro_f1": round(macro_f1, 4),
    }
    class_names = ["real", "fake", "nei"]
    for i in range(nc):
        m[f"{prefix}_f1_{class_names[i]}"] = round(float(per_f1[i]) if i < len(per_f1) else 0.0, 4)
    return m


def run_epoch(model, loader, loss_fn, optimizer, scheduler, device, config, train):
    model.train() if train else model.eval()
    total_loss, n_batches = 0.0, 0
    y_true_all, y_pred_all = [], []
    gate_means = []  # track gate statistics during eval

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        pbar = tqdm(loader, desc="  train" if train else "  eval ", leave=False)
        for batch in pbar:
            text_feat, image_feat, labels = batch
            text_feat  = text_feat.to(device)
            image_feat = image_feat.to(device)
            labels     = labels.to(device)

            logits, gate_vals = model(text_feat, image_feat)
            loss = loss_fn(logits, labels)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss: {loss.item()}")

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), config["training"]["grad_clip"])
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            else:
                gate_means.append(gate_vals.mean().item())

            total_loss += loss.item()
            n_batches  += 1
            preds = logits.argmax(dim=-1).cpu().numpy()
            y_pred_all.extend(preds.tolist())
            y_true_all.extend(labels.cpu().numpy().tolist())
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    mean_loss = total_loss / max(1, n_batches)
    avg_gate  = float(np.mean(gate_means)) if gate_means else None
    return mean_loss, np.array(y_true_all), np.array(y_pred_all), avg_gate

print("Training functions defined.")

In [ ]:
import mlflow

mlflow_enabled = False
try:
    mlflow.set_tracking_uri(CONFIG["paths"]["mlflow_dir"].as_uri())
    mlflow.set_experiment(CONFIG["mlflow"]["experiment_name"])
    _run = mlflow.start_run(run_name=run_name)
    mlflow.log_params({
        "arch_tag":      CONFIG["model"]["arch_tag"],
        "text_dim":      CONFIG["model"]["text_dim"],
        "image_dim":     CONFIG["model"]["image_dim"],
        "fusion_hidden": CONFIG["model"]["fusion_hidden"],
        "text_residual": CONFIG["model"]["text_residual"],
        "batch_size":    CONFIG["training"]["batch_size"],
        "max_epochs":    CONFIG["training"]["max_epochs"],
        "lr":            CONFIG["training"]["lr"],
        "seed":          CONFIG["training"]["seed"],
        "smoke_test":    CONFIG["safety"]["smoke_test"],
    })
    mlflow_enabled = True
    print(f"MLflow run: {_run.info.run_id}")
except Exception as e:
    print(f"MLflow disabled ({e})")

In [ ]:
best_val_f1  = -1.0
best_epoch   = -1
best_metrics = {}
patience_ctr = 0
history      = []
max_epochs   = CONFIG["safety"]["smoke_epochs"] if CONFIG["safety"]["smoke_test"] else CONFIG["training"]["max_epochs"]

print(f"Training for up to {max_epochs} epochs (patience={CONFIG['training']['patience']})")
print(f"Checkpoint dir: {run_dir}")

for epoch in range(1, max_epochs + 1):
    train_loss, yt, yp, _ = run_epoch(
        head, loaders["train"], loss_fn, optimizer, scheduler, device, CONFIG, train=True
    )
    val_loss, yv_t, yv_p, avg_gate = run_epoch(
        head, loaders["dev"], loss_fn, None, None, device, CONFIG, train=False
    )

    train_m = compute_metrics(yt,   yp,   "train", CONFIG["model"]["num_classes"])
    val_m   = compute_metrics(yv_t, yv_p, "val",   CONFIG["model"]["num_classes"])
    row     = {"epoch": epoch, "train_loss": round(train_loss,4),
               "val_loss": round(val_loss,4), "avg_gate": round(avg_gate, 3) if avg_gate else None,
               **train_m, **val_m}
    history.append(row)

    if mlflow_enabled:
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss,
                             "avg_gate": avg_gate or 0.0, **train_m, **val_m}, step=epoch)

    val_f1 = val_m["val_macro_f1"]
    gate_str = f"  gate={avg_gate:.3f}" if avg_gate is not None else ""
    marker = ""
    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_epoch   = epoch
        best_metrics = val_m
        patience_ctr = 0
        save_checkpoint(best_ckpt_path, head, epoch, val_m, CONFIG, history, is_best=True)
        marker = " ← best"
    else:
        patience_ctr += 1

    print(
        f"Epoch {epoch:02d}/{max_epochs} | "
        f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f} | "
        f"val_macro_f1={val_f1:.4f}  val_acc={val_m['val_accuracy']:.4f}"
        f"{gate_str}{marker}"
    )

    if patience_ctr >= CONFIG["training"]["patience"]:
        print(f"Early stopping at epoch {epoch} (patience={CONFIG['training']['patience']})")
        break

write_manifest(run_dir, best_ckpt_path, best_epoch, best_metrics, CONFIG)
if mlflow_enabled:
    mlflow.log_metrics({f"best_{k}": v for k, v in best_metrics.items()})
    mlflow.end_run()

print(f"\nBest epoch: {best_epoch}  |  val macro-F1: {best_val_f1:.4f}")
print(f"Best val metrics: {best_metrics}")

## Step 10: Test Evaluation, Results Export, and Gate Analysis

In [ ]:
ckpt = torch.load(best_ckpt_path, map_location=device)
head.load_state_dict(ckpt["model_state_dict"])
head.eval()

test_loss, yt_t, yt_p, avg_gate_test = run_epoch(
    head, loaders["test"], loss_fn, None, None, device, CONFIG, train=False
)
test_m = compute_metrics(yt_t, yt_p, "test", CONFIG["model"]["num_classes"])

print(f"\nTest results:")
for k, v in test_m.items():
    print(f"  {k}: {v}")
print(f"  avg_gate (test): {avg_gate_test:.3f}  (>0.5 = text wins; <0.5 = image wins)")

# ── Confusion matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(yt_t, yt_p, labels=[0,1,2])
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", ax=ax,
            xticklabels=["Real","Fake","NEI"],
            yticklabels=["Real","Fake","NEI"],
            cmap="Blues")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(
    f"Stage 05-B: Asym Gated Fusion\n"
    f"Test macro-F1={test_m['test_macro_f1']:.4f}  gate={avg_gate_test:.3f}"
)
fig.tight_layout()
cm_path = CONFIG["paths"]["results_dir"] / f"{run_name}_confusion_matrix.png"
fig.savefig(cm_path, dpi=150)
plt.show()
print(f"Saved: {cm_path}")

# ── Export results JSON ───────────────────────────────────────────────────────
results = {
    "notebook":      "05b_asymmetric_gated_fusion.ipynb",
    "proposal":      "B — Asymmetric Gated Fusion with Text Residual",
    "run_name":      run_name,
    "best_epoch":    best_epoch,
    "val_metrics":   best_metrics,
    "test_metrics":  test_m,
    "avg_gate_test": avg_gate_test,
    "arch": {
        "tag":           CONFIG["model"]["arch_tag"],
        "text_encoder":  "PhoBERT-base-v2 (frozen, Stage 3.9 CLS)",
        "image_encoder": "COOLANT image_aligned_clip (frozen, Stage 2, mean-pooled)",
        "fusion":        "asymmetric gate + text residual",
        "text_residual": CONFIG["model"]["text_residual"],
    },
    "training_history": history,
}
results_path = CONFIG["paths"]["results_dir"] / f"{run_name}_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved: {results_path}")

## Step 11: Training Curves + Gate Evolution

In [ ]:
df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(df["epoch"], df["train_loss"], label="train")
axes[0].plot(df["epoch"], df["val_loss"],   label="val")
axes[0].axvline(best_epoch, color="red", linestyle="--", label=f"best (ep {best_epoch})")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(df["epoch"], df["train_macro_f1"], label="train")
axes[1].plot(df["epoch"], df["val_macro_f1"],   label="val")
axes[1].axvline(best_epoch, color="red", linestyle="--")
axes[1].set_title("Macro F1"); axes[1].legend()

if "avg_gate" in df.columns:
    axes[2].plot(df["epoch"], df["avg_gate"], label="gate mean (val)")
    axes[2].axhline(0.5, color="gray", linestyle=":", label="0.5 (equal weight)")
    axes[2].axvline(best_epoch, color="red", linestyle="--")
    axes[2].set_title("Gate mean (>0.5 = text wins)")
    axes[2].legend()
else:
    axes[2].set_visible(False)

fig.suptitle(f"Stage 05-B: Asymmetric Gated Fusion — {run_name}")
fig.tight_layout()
curve_path = CONFIG["paths"]["results_dir"] / f"{run_name}_curves.png"
fig.savefig(curve_path, dpi=150)
plt.show()
print(f"Saved: {curve_path}")
print(f"\nNote: If gate mean stays >> 0.5, the model relies heavily on text (expected for NEI).")
print(f"      If gate mean drops toward 0 for Fake class, image consistency is discriminative.")